In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# COMMAND ----------
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
bronze_path = f"{s3_root_path}/Bronze/Nisarga/constructors"
silver_path = f"{s3_root_path}/Silver/Nisarga/constructors"
 
print("Bronze Path :", bronze_path)
print("Silver Path :", silver_path)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 1 - READ BRONZE CONSTRUCTORS DATA
# ============================================================
 
constructors_df = (
    spark.read
    .format("delta")
    .load(bronze_path)
)
 
display(constructors_df)
 
 
# COMMAND ----------
 
from pyspark.sql import functions as F
 
 
# COMMAND ----------
 
# ============================================================
# STEP 2 - SELECT REQUIRED COLUMNS
# ============================================================
# URL is excluded because it is not required for analytics.
 
constructors_selected_df = constructors_df.select(
    F.col("constructorId"),
    F.col("name"),
    F.col("nationality"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)
 
display(constructors_selected_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 3 - STANDARDIZE COLUMN NAMES
# ============================================================
 
constructors_renamed_df = (
    constructors_selected_df
    .withColumnRenamed("constructorId", "constructor_id")
)
 
display(constructors_renamed_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 4 - REMOVE NULL BUSINESS KEYS
# ============================================================
 
constructors_valid_df = (
    constructors_renamed_df
    .filter(F.col("constructor_id").isNotNull())
)
 
display(constructors_valid_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 5 - REMOVE DUPLICATES
# ============================================================
 
constructors_distinct_df = (
    constructors_valid_df
    .dropDuplicates(["constructor_id"])
)
 
display(constructors_distinct_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 6 - STANDARDIZE TEXT VALUES
# ============================================================
 
constructors_final_df = (
    constructors_distinct_df
    .withColumn(
        "name",
        F.initcap(F.col("name"))
    )
    .withColumn(
        "nationality",
        F.initcap(F.col("nationality"))
    )
)
 
display(constructors_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 7 - WRITE TO SILVER
# ============================================================
 
(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# STEP 8 - VALIDATE SILVER DATA
# ============================================================
 
silver_constructors_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)
 
display(silver_constructors_df)
 
print("Bronze Records :", constructors_df.count())
print("Silver Records :", silver_constructors_df.count())
 
print("Silver Constructors data successfully written to:")
print(silver_path)